In [1]:
!pip install -q langchain langchain-community langchain-huggingface chromadb langchain-chroma sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7

In [2]:
pip install langchain-core

In [3]:
import json
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Load your JSON data
with open("/content/complete_windows.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 2. Build the list of Document objects
docs = []
for item in data:
    text = f"Content:{item['Content']}".strip()
    docs.append(Document(
        page_content=text,
        metadata={
            "source": item.get("source", ""),
            "id": item.get("id", "")
        }
    ))

# 3. Initialize HuggingFaceEmbeddings (using model_name)
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

# 4. Batch ingestion into Chroma
batch_size = 100
db = None

for i in range(0, len(docs), batch_size):
    print(f"Processing {i} -> {i + batch_size}")
    batch = docs[i:i + batch_size]

    if db is None:
        # Initialize the database with the first batch
        db = Chroma.from_documents(
            documents=batch,
            embedding=embeddings,
            persist_directory="db6_v6"
        )
    else:
        # Incrementally add subsequent batches to the existing database
        db.add_documents(documents=batch)

/tmp/ipykernel_1548/2951088611.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
/tmp/ipykernel_1548/2951088611.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Processing 0 -> 100
Processing 100 -> 200
Processing 200 -> 300
Processing 300 -> 400
Processing 400 -> 500


In [4]:
!zip -r db6_v6 /content/db6_v6

  adding: content/db6_v6/ (stored 0%)
  adding: content/db6_v6/d8693f84-a87d-4de1-9995-8b65be0e3c9c/ (stored 0%)
  adding: content/db6_v6/d8693f84-a87d-4de1-9995-8b65be0e3c9c/data_level0.bin (deflated 100%)
  adding: content/db6_v6/d8693f84-a87d-4de1-9995-8b65be0e3c9c/header.bin (deflated 63%)
  adding: content/db6_v6/d8693f84-a87d-4de1-9995-8b65be0e3c9c/link_lists.bin (stored 0%)
  adding: content/db6_v6/d8693f84-a87d-4de1-9995-8b65be0e3c9c/length.bin (deflated 71%)
  adding: content/db6_v6/chroma.sqlite3 (deflated 62%)


In [5]:
from google.colab import files
files.download("/content/db6_v6.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>